# Exercise 30: The Number of Employees Which Report to Each Employee

**Source:** [LeetCode 1731](https://leetcode.com/problems/the-number-of-employees-which-report-to-each-employee/) (Easy).

Table: `Employees`
```
+-------------+----------+
| Column Name | Type     |
+-------------+----------+
| employee_id | int      |
| name        | varchar  |
| reports_to  | int      |
| age         | int      |
+-------------+----------+
```

- `employee_id` is unique.
- `reports_to` is the `employee_id` of the employee's manager (`NULL` if they have none).

**Task.** A *manager* is anyone with at least one direct report. For every manager, show `employee_id`, `name`, the number of direct reports (`reports_count`) and their average age rounded to a whole number (`average_age`), sorted by `employee_id`.

Connect to the database `postgres` using the username `postgres`.

In [1]:
%load_ext sql
%sql postgresql+psycopg://postgres@localhost:5432/postgres

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'postgresql+psycopg://postgres@localhost:5432/postgres'

### Create new database for the exercise

Make sure first that a database with the same name is removed in case it already exists.

In [2]:
%sql DROP DATABASE IF EXISTS exercise_30 WITH (FORCE);
%sql CREATE DATABASE exercise_30;

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

Connect to the database.

*Caution.* No semicolon at the end otherwise it belongs to the name of the database.

In [3]:
%sql postgresql+psycopg://postgres@localhost:5432/exercise_30 --alias exercise_30

Connecting and switching to connection 'exercise_30'

#### Fill the database with content

In [4]:
%%sql
DROP TABLE IF EXISTS Employees;

CREATE TABLE Employees (
    employee_id INTEGER UNIQUE,
    name VARCHAR,
    reports_to INTEGER,
    age INTEGER
);

INSERT INTO Employees (employee_id, name, reports_to, age)
VALUES
(9, 'Hercy', NULL, 43),
(6, 'Alice', 9   , 41),
(4, 'Bob', 9   , 36),
(2, 'Winston', NULL, 37);

SELECT * FROM Employees;

Running query in 'exercise_30'

4 rows affected.

4 rows affected.

employee_id,name,reports_to,age
9,Hercy,None,43
6,Alice,9,41
4,Bob,9,36
2,Winston,None,37


#### Solve the exercise

**Task.**
A *manager* is anyone with at least one direct report. For every manager, show `employee_id`, `name`, the number of direct reports (`reports_count`) and their average age rounded to a whole number (`average_age`), sorted by `employee_id`.

In [5]:
%%sql
SELECT
M.employee_id, M.name, COUNT(E.*) AS reports_count, ROUND(AVG(E.age), 0) AS average_age
FROM Employees AS M, Employees AS E
WHERE M.employee_id = E.reports_to
GROUP BY M.employee_id, M.name
ORDER BY M.employee_id;

Running query in 'exercise_30'

1 rows affected.

employee_id,name,reports_count,average_age
9,Hercy,2,39


#### Explanation

The exercise can be solved using a self join.
- Consider one table `Employees` as the managers `M`, and the same table again as the employees `E`.
- Group `M` by `employee_id` and `name`.
- Only select those rows where `M.employee_id = E.reports_to`, i.e. where there is someone reporting to `M.employee_id`.
- Add to the `M.employee_id` and `M.name` the count over all entries in `E` such that `M.employee_id = E.reports_to`
- and take the average age rounded to an integer.
- Order by `M.employee_id`

#### Remark
Another possibility is to use the `WITH` keyword to define a new table 'before' querying:

In [10]:
%%sql

WITH Manager AS (
    SELECT reports_to AS manager_id,
    COUNT(*) AS reports_count,
    ROUND(AVG(age), 0) AS average_age
    FROM Employees
    GROUP BY reports_to
    HAVING reports_to IS NOT NULL
)
SELECT Employees.employee_id, Employees.name, Manager.reports_count, Manager.average_age
FROM Manager LEFT JOIN Employees
ON Employees.employee_id = Manager.manager_id
ORDER BY Employees.employee_id;

Running query in 'exercise_30'

1 rows affected.

employee_id,name,reports_count,average_age
9,Hercy,2,39


We note that in Leet the second code snippet has a remarkably faster runtime.

#### Close connection

In [11]:
%sql --close exercise_30
%sql postgresql+psycopg://postgres@localhost:5432/postgres
%sql DROP DATABASE IF EXISTS exercise_30;

Switching to connection 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

**Well done!** (approved by Leet)